# Phase 1 - YOLO11n Kaggle Training and Evaluation Notebook

## Kaggle Workflow

このノートブックは Kaggle GPU 環境専用です。
Codex のプロジェクトで管理しているデータセットと `best.pt` を Kaggle Dataset として追加して使用します。

現在の目的は、学習済みモデルを現在のデプロイ設定で評価できる構造を維持することです。
学習セルは再現用の training record として残し、再学習が必要な場合だけ明示的に実行します。

1. Kaggle 環境と共通定数を準備する
2. `data.yaml` を明示的に選択し、Kaggle 用の絶対パスへ正規化する
3. train / validation / test とクラス定義を検証する
4. 学習済み `best.pt` を Kaggle Input から明示的に選択する
5. `imgsz=1024`、`conf_threshold=0.5` で test split evaluation を実行する
6. evaluation artifacts を ZIP にまとめてダウンロードする

In [ ]:
# 手順1: Kaggle 環境と共通定数を準備する
!pip install -U ultralytics

from pathlib import Path
import json
import math
import shutil
import zipfile

import pandas as pd
import yaml
from ultralytics import YOLO

try:
    import torch
    gpu_available = torch.cuda.is_available()
    gpu_name = torch.cuda.get_device_name(0) if gpu_available else "CPU"
except Exception:
    gpu_available = False
    gpu_name = "CPU"

print(f"GPU available: {gpu_available}")
print(f"GPU name: {gpu_name}")

KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORK_DIR = Path("/kaggle/working")
NORMALIZED_DATA_YAML_PATH = KAGGLE_WORK_DIR / "receipt_detection_data.yaml"
EVALUATION_ROOT = KAGGLE_WORK_DIR / "evaluation"
ARTIFACT_DIR = KAGGLE_WORK_DIR / "receipt_detection_artifacts"

for directory in [EVALUATION_ROOT, ARTIFACT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

TRAIN_IMGSZ = 640
DEPLOY_IMGSZ = 1024
DEPLOY_CONF_THRESHOLD = 0.5
EXPECTED_CLASS_NAMES = ["date", "phone", "total"]


def read_box_metric(metrics, attr_name, result_key):
    # 検証結果からボックス系メトリクスを読み出す
    box_metrics = getattr(metrics, "box", None)
    if box_metrics is not None and hasattr(box_metrics, attr_name):
        value = getattr(box_metrics, attr_name)
        if value is not None:
            return float(value)

    results_dict = getattr(metrics, "results_dict", {})
    if isinstance(results_dict, dict) and result_key in results_dict:
        return float(results_dict[result_key])

    return float("nan")

In [ ]:
# 手順2: Kaggle Input から data.yaml を選択して絶対パスへ正規化する

DATA_YAML_SELECTION = None


def select_data_yaml(root_dir, selection=None):
    # 候補が複数ある場合は自動選択せず、番号の指定を要求する
    candidates = sorted(root_dir.rglob("data.yaml"))
    if not candidates:
        raise FileNotFoundError("No data.yaml was found under /kaggle/input.")

    print("data.yaml candidates:")
    for index, candidate in enumerate(candidates, start=1):
        print(f"{index}: {candidate}")

    if len(candidates) == 1:
        return candidates[0]

    if selection is None:
        raise RuntimeError(
            "Multiple data.yaml files were found. "
            "Set DATA_YAML_SELECTION to the displayed candidate number and run this cell again."
        )

    selected_index = int(selection) - 1
    if not 0 <= selected_index < len(candidates):
        raise IndexError("DATA_YAML_SELECTION is outside the candidate range.")
    return candidates[selected_index]


def choose_existing_split(split_root, candidates, split_name):
    # Roboflow の split ディレクトリ候補から実在するパスを選ぶ
    for relative_path in candidates:
        resolved = split_root / relative_path
        if resolved.exists():
            return str(resolved.resolve())

    raise FileNotFoundError(
        f"Could not find a valid path for {split_name}: {candidates}"
    )


source_data_yaml_path = select_data_yaml(
    KAGGLE_INPUT_DIR,
    DATA_YAML_SELECTION,
)

with source_data_yaml_path.open("r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)

split_root = source_data_yaml_path.parent
data_config["train"] = choose_existing_split(
    split_root,
    ["train/images", "train/image"],
    "train",
)
data_config["val"] = choose_existing_split(
    split_root,
    ["valid/images", "val/images", "validation/images"],
    "validation",
)
data_config["test"] = choose_existing_split(
    split_root,
    ["test/images"],
    "test",
)

with NORMALIZED_DATA_YAML_PATH.open("w", encoding="utf-8") as file:
    yaml.safe_dump(
        data_config,
        file,
        sort_keys=False,
        allow_unicode=True,
    )

DATA_YAML_PATH = NORMALIZED_DATA_YAML_PATH

print(f"Selected data.yaml: {source_data_yaml_path}")
print(f"Normalized data.yaml: {DATA_YAML_PATH}")
print(f"train: {data_config['train']}")
print(f"val:   {data_config['val']}")
print(f"test:  {data_config['test']}")
print(f"names: {data_config.get('names')}")

In [ ]:
# 手順3: 正規化した data.yaml とデータセットの健全性を確認する

import math
from pathlib import Path
from PIL import Image as PILImage
import yaml


def normalize_class_names(names_field):
    if isinstance(names_field, list):
        return [str(name) for name in names_field]

    if isinstance(names_field, dict):
        try:
            sorted_items = sorted(
                names_field.items(),
                key=lambda item: int(item[0])
            )
        except Exception:
            sorted_items = sorted(
                names_field.items(),
                key=lambda item: str(item[0])
            )

        return [str(name) for _, name in sorted_items]

    raise TypeError("The names field in data.yaml must be a list or dict.")


def count_split_contents(images_path_str):
    img_dir = Path(images_path_str)

    label_dir = img_dir.parent.parent / "labels" / img_dir.name

    if not label_dir.exists():
        label_dir = img_dir.parent / "labels"

    img_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    img_count = (
        sum(1 for f in img_dir.iterdir() if f.suffix.lower() in img_exts)
        if img_dir.exists()
        else 0
    )

    lbl_count = (
        sum(1 for f in label_dir.iterdir() if f.suffix == ".txt")
        if label_dir.exists()
        else 0
    )

    return img_count, lbl_count, label_dir



def check_annotation_quality(label_dir, class_names, split_name):

    issues = []

    class_counts = {
        i: 0 for i in range(len(class_names))
    }

    total_box_heights = []

    label_dir = Path(label_dir)

    if not label_dir.exists():
        return (
            [f"{split_name}: label directory not found: {label_dir}"],
            class_counts,
            total_box_heights
        )


    for lbl_file in sorted(label_dir.glob("*.txt")):

        lines = lbl_file.read_text(
            encoding="utf-8"
        ).strip().splitlines()


        if not lines:
            issues.append(
                f"EMPTY_LABEL [{split_name}]: {lbl_file.name}"
            )
            continue


        seen_classes = set()


        for line in lines:

            parts = line.strip().split()


            # YOLO detection format must have 5 values
            if len(parts) != 5:
                issues.append(
                    f"BAD_FORMAT [{split_name}]: {lbl_file.name} -> {line}"
                )
                continue


            cls = int(parts[0])


            try:
                cx, cy, w, h = map(
                    float,
                    parts[1:]
                )

            except ValueError:
                issues.append(
                    f"BAD_VALUE [{split_name}]: {lbl_file.name}"
                )
                continue



            # coordinate range check
            for val, name in [
                (cx, "cx"),
                (cy, "cy"),
                (w, "w"),
                (h, "h")
            ]:

                if not (0.0 <= val <= 1.0):

                    issues.append(
                        f"OUT_OF_RANGE [{split_name}] "
                        f"{lbl_file.name} {name}={val:.4f}"
                    )


            # very small box
            if h < 0.005:

                issues.append(
                    f"ZERO_HEIGHT [{split_name}] "
                    f"{lbl_file.name} h={h:.6f}"
                )


            # total class check
            if cls == 2:

                total_box_heights.append(
                    (h, lbl_file.name)
                )


                if h > 0.5:

                    issues.append(
                        f"TOTAL_FULL_RECEIPT [{split_name}] "
                        f"{lbl_file.name} "
                        f"w={w:.3f} h={h:.3f}"
                    )


            # duplicate class check
            if cls in seen_classes:

                issues.append(
                    f"DUPLICATE_CLASS [{split_name}] "
                    f"{lbl_file.name} class {cls}"
                )

            seen_classes.add(cls)


            if cls < len(class_names):
                class_counts[cls] += 1



    return issues, class_counts, total_box_heights



def check_image_resolution(images_path_str, sample_count=5):

    img_dir = Path(images_path_str)

    img_exts = {
        ".jpg",
        ".jpeg",
        ".png"
    }

    images = [
        f for f in img_dir.iterdir()
        if f.suffix.lower() in img_exts
    ]


    sizes = set()


    for img_path in images[:sample_count]:

        with PILImage.open(img_path) as im:
            sizes.add(im.size)


    return sizes



# ===============================
# Load Kaggle normalized data.yaml
# ===============================

if "DATA_YAML_PATH" not in globals():

    raise NameError(
        "Run the previous cell first to create DATA_YAML_PATH."
    )


with DATA_YAML_PATH.open(
    "r",
    encoding="utf-8"
) as file:

    data_config = yaml.safe_load(file)



required_keys = [
    "train",
    "val",
    "test",
    "names"
]


missing_keys = [
    key for key in required_keys
    if key not in data_config
]


if missing_keys:

    raise KeyError(
        f"data.yaml missing keys: {missing_keys}"
    )



# ===============================
# Class check
# ===============================

class_names = normalize_class_names(
    data_config["names"]
)


class_count = int(
    data_config.get(
        "nc",
        len(class_names)
    )
)


if class_count != len(class_names):

    raise ValueError(
        f"Class mismatch nc={class_count}, names={len(class_names)}"
    )


expected_class_names = EXPECTED_CLASS_NAMES


if class_names == expected_class_names:

    print("✓ Class order confirmed: date / phone / total")

else:

    print(
        f"⚠ WARNING: Expected {expected_class_names}, got {class_names}"
    )



# ===============================
# Dataset split check
# ===============================

print("\n=== Dataset split check ===")


all_issues = []

total_images = 0


all_class_counts = {
    i: 0 for i in range(len(class_names))
}


all_total_heights = []



for split_name, split_path in [
    ("train", data_config["train"]),
    ("val", data_config["val"]),
    ("test", data_config["test"])
]:


    if not Path(split_path).exists():

        raise FileNotFoundError(
            f"{split_name} path missing: {split_path}"
        )


    img_count, lbl_count, lbl_dir = count_split_contents(
        split_path
    )


    total_images += img_count


    status = "✓" if img_count > 0 and lbl_count > 0 else "✗"


    print(
        f"[{status}] {split_name}: "
        f"{img_count} images, {lbl_count} labels"
    )



    # resolution check
    resolutions = check_image_resolution(
        split_path
    )


    for size in resolutions:

        if size[0] <= 640 or size[1] <= 640:

            print(
                f"⚠ {split_name} image resolution: {size}"
            )



    issues, counts, heights = check_annotation_quality(
        lbl_dir,
        class_names,
        split_name
    )


    all_issues.extend(issues)

    all_total_heights.extend(
        heights
    )


    for k, v in counts.items():

        all_class_counts[k] += v




print(
    f"\nTotal images: {total_images}"
)



# ===============================
# Class distribution
# ===============================

print(
    "\n=== Class annotation counts ==="
)


for cls_id, cls_name in enumerate(class_names):

    count = all_class_counts[cls_id]

    pct = (
        count / total_images * 100
        if total_images > 0
        else 0
    )


    print(
        f"class {cls_id} ({cls_name}): "
        f"{count}/{total_images} ({pct:.0f}%)"
    )



# ===============================
# Total bbox check
# ===============================

large_h = 0


if all_total_heights:

    heights = [
        h for h, _ in all_total_heights
    ]


    large_h = sum(
        1 for h in heights
        if h >= 0.5
    )


    print(
        "\n=== total bbox height distribution ==="
    )

    print(
        f"Count: {len(heights)}"
    )

    print(
        f"Min: {min(heights):.4f}, "
        f"Max: {max(heights):.4f}, "
        f"Avg: {sum(heights)/len(heights):.4f}"
    )

    print(
        f"Suspicious full receipt boxes: {large_h}"
    )



# ===============================
# Issue summary
# ===============================

print(
    f"\n=== Annotation issues ({len(all_issues)}) ==="
)


if not all_issues:

    print(
        "✓ No annotation issues found."
    )

else:

    for issue in all_issues:

        print(issue)



if all_issues or large_h > 0:

    print(
        "\n⚠ Dataset issues detected."
    )

else:

    print(
        "\n✓ Dataset health check passed."
    )



print("\nDataset configuration:")
print(
    f"train: {data_config['train']}"
)
print(
    f"val:   {data_config['val']}"
)
print(
    f"test:  {data_config['test']}"
)
print(
    f"nc: {class_count}, names: {class_names}"
)

In [7]:
# 手順4: YOLO11n の training record（再学習時のみ実行する）
# 現在の evaluation 作業では、このセルを実行しない

from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data=DATA_YAML_PATH,

    # 基本学習設定
    epochs=300,
    imgsz=TRAIN_IMGSZ,
    batch=8,
    device=0,
    patience=80,

    # レシート向けの軽いデータ拡張
    mosaic=0.2,
    mixup=0.0,
    cutmix=0.0,

    degrees=0.0,
    translate=0.05,
    scale=0.2,
    shear=0.0,
    perspective=0.0,

    # レシートの向きを維持するため反転しない
    fliplr=0.0,
    flipud=0.0,

    # 再現性を維持する
    deterministic=True,
    seed=42,

    # 学習結果の保存先
    project="/kaggle/working/runs/detect",
    name="receipt_yolo11n_final",
    exist_ok=False,

    # 検証結果を保存する
    val=True,
    plots=True
)

Ultralytics 8.4.117 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=0.2, multi_scale=0.0, name=receipt_yolo11n_final, nbs=64, nms=

## Formal Deployment Evaluation

このセクションは Kaggle Dataset としてマウントした現在の正式モデル `best.pt` を評価する。
Kaggle セッション内で生成した学習出力には依存しない。

- split: `test`
- imgsz: `1024`
- conf: `0.5`
- training: 実施しない

In [ ]:
# 手順5: Kaggle Input から正式な best.pt artifact を明示的に選択する

import hashlib


BEST_PT_SELECTION = None
EVAL_SPLIT = "test"
EVAL_NAME = "test_1024_conf_0_5"
EVAL_DIR = EVALUATION_ROOT / EVAL_NAME


def select_best_pt(root_dir, selection=None):
    # 候補が複数ある場合は自動選択せず、番号の指定を要求する
    candidates = sorted(root_dir.rglob("best.pt"))
    if not candidates:
        raise FileNotFoundError(
            "No best.pt was found under /kaggle/input. "
            "Mount the current models/best.pt as a Kaggle Dataset."
        )

    print("best.pt candidates:")
    for index, candidate in enumerate(candidates, start=1):
        print(f"{index}: {candidate}")

    if len(candidates) == 1:
        return candidates[0]

    if selection is None:
        raise RuntimeError(
            "Multiple best.pt files were found. "
            "Set BEST_PT_SELECTION to the displayed candidate number and run this cell again."
        )

    selected_index = int(selection) - 1
    if not 0 <= selected_index < len(candidates):
        raise IndexError("BEST_PT_SELECTION is outside the candidate range.")
    return candidates[selected_index]


def calculate_sha256(file_path):
    # 評価に使用するモデルファイルの SHA256 を計算する
    digest = hashlib.sha256()
    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


if DEPLOY_IMGSZ != 1024:
    raise ValueError(f"DEPLOY_IMGSZ must be 1024, got {DEPLOY_IMGSZ}.")
if DEPLOY_CONF_THRESHOLD != 0.5:
    raise ValueError(
        "DEPLOY_CONF_THRESHOLD must be 0.5, "
        f"got {DEPLOY_CONF_THRESHOLD}."
    )

BEST_PT_PATH = select_best_pt(KAGGLE_INPUT_DIR, BEST_PT_SELECTION)
DATA_YAML_PATH = NORMALIZED_DATA_YAML_PATH
BEST_PT_SHA256 = calculate_sha256(BEST_PT_PATH)
best_model = YOLO(str(BEST_PT_PATH))

print(f"Selected model artifact: {BEST_PT_PATH}")
print(f"Model SHA256: {BEST_PT_SHA256}")
print(f"Evaluation data: {DATA_YAML_PATH}")
print(f"Evaluation output: {EVAL_DIR}")
print(f"Evaluation split: {EVAL_SPLIT}")
print(f"Deployment imgsz: {DEPLOY_IMGSZ}")
print(f"Deployment confidence threshold: {DEPLOY_CONF_THRESHOLD}")

In [ ]:
# 手順6: 正式なデプロイ条件で test split evaluation を実行する


def metric_values_to_list(values):
    # NumPy 配列や tensor を JSON 保存可能な float リストへ変換する
    if values is None:
        return []
    if hasattr(values, "tolist"):
        values = values.tolist()
    if not isinstance(values, list):
        values = [values]
    return [float(value) for value in values]


def resolve_class_name(names, class_id):
    # モデルが保持するクラス名から対象クラスの名称を取得する
    if isinstance(names, dict):
        return str(names.get(class_id, names.get(str(class_id), class_id)))
    return str(names[class_id])


test_results = best_model.val(
    data=str(DATA_YAML_PATH),
    split=EVAL_SPLIT,
    imgsz=DEPLOY_IMGSZ,
    conf=DEPLOY_CONF_THRESHOLD,
    project=str(EVALUATION_ROOT),
    name=EVAL_NAME,
    exist_ok=True,
    plots=True,
    verbose=False,
)

EVAL_DIR = Path(test_results.save_dir)
box_metrics = test_results.box

precision = read_box_metric(test_results, "mp", "metrics/precision(B)")
recall = read_box_metric(test_results, "mr", "metrics/recall(B)")
map50 = read_box_metric(test_results, "map50", "metrics/mAP50(B)")
map50_95 = read_box_metric(test_results, "map", "metrics/mAP50-95(B)")

class_ids = metric_values_to_list(getattr(box_metrics, "ap_class_index", []))
class_ids = [int(class_id) for class_id in class_ids]
class_precision = metric_values_to_list(getattr(box_metrics, "p", []))
class_recall = metric_values_to_list(getattr(box_metrics, "r", []))
class_map50 = metric_values_to_list(getattr(box_metrics, "ap50", []))
class_map50_95 = metric_values_to_list(getattr(box_metrics, "ap", []))

per_class_metrics = []
for index, class_id in enumerate(class_ids):
    per_class_metrics.append(
        {
            "class_id": class_id,
            "class_name": resolve_class_name(best_model.names, class_id),
            "precision": class_precision[index],
            "recall": class_recall[index],
            "mAP50": class_map50[index],
            "mAP50-95": class_map50_95[index],
        }
    )

metrics_data = {
    "overall": {
        "precision": precision,
        "recall": recall,
        "mAP50": map50,
        "mAP50-95": map50_95,
    },
    "per_class": per_class_metrics,
}

config_data = {
    "model": {
        "artifact_path": str(BEST_PT_PATH),
        "sha256": BEST_PT_SHA256,
    },
    "dataset": {
        "data_yaml": str(DATA_YAML_PATH),
        "split": EVAL_SPLIT,
    },
    "inference": {
        "imgsz": DEPLOY_IMGSZ,
        "conf": DEPLOY_CONF_THRESHOLD,
    },
    "classes": {
        str(class_id): resolve_class_name(best_model.names, class_id)
        for class_id in (
            sorted(best_model.names)
            if isinstance(best_model.names, dict)
            else range(len(best_model.names))
        )
    },
}

metrics_path = EVAL_DIR / "metrics.json"
config_path = EVAL_DIR / "config.json"
sha256_path = EVAL_DIR / "best_pt_sha256.txt"
confusion_matrix_path = EVAL_DIR / "confusion_matrix.png"

with metrics_path.open("w", encoding="utf-8") as file:
    json.dump(metrics_data, file, indent=2, ensure_ascii=False)

with config_path.open("w", encoding="utf-8") as file:
    json.dump(config_data, file, indent=2, ensure_ascii=False)

sha256_path.write_text(BEST_PT_SHA256 + "\n", encoding="utf-8")

if not confusion_matrix_path.exists():
    raise FileNotFoundError(
        f"confusion_matrix.png was not generated: {confusion_matrix_path}"
    )

print("Overall test metrics:")
print(f" - Precision: {precision:.4f}")
print(f" - Recall:    {recall:.4f}")
print(f" - mAP50:     {map50:.4f}")
print(f" - mAP50-95:  {map50_95:.4f}")

print("\nPer-class metrics:")
for class_metrics in per_class_metrics:
    print(
        f" - {class_metrics['class_name']}: "
        f"P={class_metrics['precision']:.4f}, "
        f"R={class_metrics['recall']:.4f}, "
        f"mAP50={class_metrics['mAP50']:.4f}, "
        f"mAP50-95={class_metrics['mAP50-95']:.4f}"
    )

print(f"\nEvaluation directory: {EVAL_DIR}")
print(f"Metrics JSON: {metrics_path}")
print(f"Evaluation config: {config_path}")
print(f"Confusion matrix: {confusion_matrix_path}")
print(f"Model SHA256: {BEST_PT_SHA256}")

In [ ]:
# 手順7: deployment evaluation artifact を ZIP にまとめる

ARTIFACT_ZIP_PATH = KAGGLE_WORK_DIR / "test_1024_conf_0_5_evaluation_artifacts.zip"
if ARTIFACT_ZIP_PATH.exists():
    ARTIFACT_ZIP_PATH.unlink()

required_artifacts = [
    EVAL_DIR / "metrics.json",
    EVAL_DIR / "config.json",
    EVAL_DIR / "confusion_matrix.png",
    EVAL_DIR / "best_pt_sha256.txt",
]
missing_artifacts = [path for path in required_artifacts if not path.exists()]
if missing_artifacts:
    raise FileNotFoundError(
        f"Required evaluation artifacts are missing: {missing_artifacts}"
    )

with zipfile.ZipFile(
    ARTIFACT_ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for artifact_path in sorted(EVAL_DIR.rglob("*")):
        if not artifact_path.is_file():
            continue
        archive_name = Path("evaluation") / EVAL_NAME / artifact_path.relative_to(EVAL_DIR)
        archive.write(artifact_path, arcname=str(archive_name))
        print(f"Added to ZIP: {archive_name}")

print(f"Evaluation artifact ZIP: {ARTIFACT_ZIP_PATH}")

In [ ]:
# 手順8: Kaggle から deployment evaluation artifact をダウンロードする

from IPython.display import FileLink, display

print("test_1024_conf_0_5_evaluation_artifacts.zip:")
display(FileLink(str(ARTIFACT_ZIP_PATH)))